## Import Libraries

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder


## Load Data

In [0]:
df = pd.read_csv('/Workspace/Users/ajiboyeniola@gmail.com/lead-scoring/data/raw/leads.csv')
df.shape

## Drop Unnecessary Columns

In [0]:
df = pd.DataFrame(df)
df = df.drop(['lead_id', 'company_name', 'created_date', 'job_title'], axis=1)

## Handle missing values

In [0]:
missing_values = pd.isnull(df).sum()
missing_values

In [0]:
df['days_since_last_contact'].fillna(df['days_since_last_contact'].median(), inplace=True)
df['budget_indicated'].fillna(0, inplace=True)
df['ad_platform'].fillna('unknown', inplace=True)

print(df.isnull().sum())


## Split Data: Stratified Split

In [0]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=['converted'])
y = df['converted']

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Confirm shapes
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Confirm stratification worked
print(f"\ny_train distribution:\n{y_train.value_counts(normalize=True)}")
print(f"\ny_test distribution:\n{y_test.value_counts(normalize=True)}")

## Encode Categorical Features

In [0]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(cat_cols)

### Ordinal encoding

In [0]:
oe = OrdinalEncoder(categories=[
    ['Small', 'Medium', 'Large'],
    ['Awareness', 'Consideration', 'Decision'],
    ['Morning', 'Afternoon', 'Evening']
])

oe.fit(X_train[['company_size', 'funnel_stage', 'preferred_contact_time']])

X_train[['company_size', 'funnel_stage', 'preferred_contact_time']] = oe.transform(X_train[['company_size', 'funnel_stage', 'preferred_contact_time']])

X_test[['company_size', 'funnel_stage', 'preferred_contact_time']] = oe.transform(X_test[['company_size', 'funnel_stage', 'preferred_contact_time']])

print(X_train['company_size'].unique())
print(X_train['funnel_stage'].unique())
print(X_train['preferred_contact_time'].unique())

### one-hot encoding

In [0]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    drop='first',
    sparse_output=False,
    handle_unknown='ignore'
)

ohe_cols = ['business_type', 'industry', 'lead_source', 'ad_platform']

ohe.fit(X_train[ohe_cols])

# Transform both
X_train_ohe = pd.DataFrame(
    ohe.transform(X_train[ohe_cols]),
    columns=ohe.get_feature_names_out(ohe_cols),
    index=X_train.index
)

X_test_ohe = pd.DataFrame(
    ohe.transform(X_test[ohe_cols]),
    columns=ohe.get_feature_names_out(ohe_cols),
    index=X_test.index
)

# Drop original columns and join encoded ones
X_train = X_train.drop(columns=ohe_cols).join(X_train_ohe)
X_test = X_test.drop(columns=ohe_cols).join(X_test_ohe)

print(X_train.shape)
print(X_test.shape)

In [0]:
print(X_test.select_dtypes(include='object').columns.tolist())
print(X_train.select_dtypes(include='object').columns.tolist())

In [0]:
display(df)

## Scaling numerical features

In [0]:
from sklearn.preprocessing import RobustScaler

# Define numerical columns to scale
num_cols = ['lead_age_days', 'email_opens', 'website_visits', 'content_downloads', 'days_since_last_contact', 'ad_clicks', 'num_contacts']

# Initialize the RobustScaler
scaler = RobustScaler()

# Fit on train only
scaler.fit(X_train[num_cols])

# Transform Both
X_train[num_cols] = scaler.transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

pd.set_option('display.max_columns', None)
print(X_train[num_cols].describe())

### Drop redundant columns

In [0]:
df = df.drop(columns=['email_clicks', 'pages_viewed'])
print(df.shape)